# IEEE-CIS Fraud Detection — Feature Engineering

## Purpose

EDA told us *what* the data looks like. Feature engineering turns those insights
into model-ready signals.

Every feature we build here is grounded in an EDA finding — no random transformations.

## Features We Will Build

| Feature | Source Insight |
|---|---|
| `log_TransactionAmt` | Amount is right-skewed — log normalises it |
| `amt_is_outlier` | IQR outlier flag — statistical unusualness is a signal |
| `hour`, `hour_is_risky` | Fraud peaks 5–9am — 4x swing |
| `day_of_week` | Weak but included |
| `card4_card6` | Interaction — discover credit is riskiest combo |
| `email_grouped` | Target encode P_emaildomain (59 values) |
| `email_is_anonymous` | protonmail/anonymous = explicit risk flag |
| `M4_encoded` | Ordinal encode M0→0, M1→1, M2→2 |
| `M_null_count` | Missing match data is itself a signal |
| `high_velocity` | C columns above 75th pct — card being hammered |
| `has_identity` | Already engineered in EDA |


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

train_transactions = pd.read_csv('data/raw/train_transaction.csv')
train_identity = pd.read_csv('data/raw/train_identity.csv')
train = pd.merge(train_transactions, train_identity, on='TransactionID', how='left')

train['has_identity'] = train['TransactionID'].isin(train_identity['TransactionID']).astype(int)

print('Data loaded:', train.shape)

Data loaded: (590540, 435)


## Chapter 1 — Transforming TransactionAmt

### Why log-transform?

From EDA we know `TransactionAmt` is heavily right-skewed — most transactions
are small but a few reach $31,937. This causes two problems:

1. **Tree models** — extreme values dominate split criteria, wasting splits on outliers
2. **Distance-based models** (KNN, SVM, neural nets) — large values dwarf everything else

`log1p(x)` = log(x + 1) — the +1 handles zeros safely.



Creating two new columns:
- `log_TransactionAmt` — log1p transform of TransactionAmt
- `amt_is_outlier` — binary flag: 1 if TransactionAmt is above the IQR upper fence, else 0

Recall the IQR upper fence formula: `Q3 + 1.5 × IQR`

In [ ]:
train['log_TransactionAmt'] = np.log1p(train['TransactionAmt'])
Q1=train['TransactionAmt'].quantile(0.25)
Q3=train['TransactionAmt'].quantile(0.75)
train['is_outlier'] = (train['TransactionAmt'] > (Q3 + 1.5 * (Q3 - Q1))).astype(int)


In [ ]:
print(train[['TransactionAmt','log_TransactionAmt','is_outlier','is_round_amount']].head(10))
print(f"\nOutliers: {train['is_outlier'].sum():,}")



   TransactionAmt  log_TransactionAmt  is_outlier  is_round_amount
0            68.5            4.241327           0                0
1            29.0            3.401197           0                1
2            59.0            4.094345           0                1
3            50.0            3.931826           0                1
4            50.0            3.931826           0                1
5            49.0            3.912023           0                1
6           159.0            5.075174           0                1
7           422.5            6.048554           1                0
8            15.0            2.772589           0                1
9           117.0            4.770685           0                1

Outliers: 66,482
Round amounts: 305,013


## Chapter 2 — Time Features

### Why
From EDA: fraud rate peaks between 5–9am (~10.5%) vs afternoon low of ~2.5% — a 4x swing.
Raw `TransactionDT` is seconds from an unknown reference point — useless directly.
We extract meaningful cycles from it.

### Features Built
- `hour` — hour of day (0–23), extracted using integer division and modulo
- `day_of_week` — day cycle (0–6), relative not absolute (reference point unknown)
- `hour_is_risky` — binary flag, 1 if hour is 5–9 inclusive, captures the fraud spike explicitly



In [9]:
train['day_of_week'] = (train['TransactionDT'] // (24*3600)) % 7
train['hour'] = (train['TransactionDT'] // 3600) % 24
train['hour_is_risky']=train['hour'].isin([5,6,7,8,9]).astype(int)

## Chapter 3 — Card Interaction Feature

### Why
From EDA: `card4` (network) and `card6` (type) each have fraud signal individually,
but the combination is more informative — discover credit (7.93%) vs mastercard debit (2.16%)
is a 3.7x difference neither column captures alone.

### Features Built
- `card4_card6` — string concatenation of card4 + card6 (e.g. "discover_credit")
  Nulls filled with "unknown" before concatenation to avoid TypeError
  Will be target encoded in Chapter 9

### Note
Concatenation chosen over a binary flag because the interaction has 8+ levels with
meaningful gradient — collapsing to 0/1 would lose that signal.


In [10]:
train['card4_6']=train['card4'].fillna('unknown') + '_' + train['card6'].fillna('unknown')

## Chapter 4 — Email Features

### Why
From EDA: `P_emaildomain` has 59 unique values with vastly different fraud rates —
protonmail.com (40.79%), mail.com (18.96%), outlook.com (9.46%) vs baseline 3.5%.
High cardinality rules out one-hot encoding.

### Features Built
- `email_target_encoded` — each domain replaced with its fraud rate (fit on train only)
  Unseen domains filled with train mean fraud rate

> **Leakage note:** Target encoding must be fit on train only and applied to val/test.
> Temporary version below uses full dataset — recomputed correctly after the split in Chapter 8.

In [12]:
email_fraud_map = train.groupby('P_emaildomain')['isFraud'].mean()
train['email_target_encoded'] = train['P_emaildomain'].map(email_fraud_map).fillna(train['isFraud'].mean())


In [13]:
email_fraud_map

P_emaildomain
aim.com             0.126984
anonymous.com       0.023217
aol.com             0.021811
att.net             0.007439
bellsouth.net       0.027763
cableone.net        0.018868
centurylink.net     0.000000
cfl.rr.com          0.000000
charter.net         0.030637
comcast.net         0.031187
cox.net             0.020818
earthlink.net       0.021401
embarqmail.com      0.034615
frontier.com        0.028571
frontiernet.net     0.025641
gmail               0.022177
gmail.com           0.043542
gmx.de              0.000000
hotmail.co.uk       0.000000
hotmail.com         0.052950
hotmail.de          0.000000
hotmail.es          0.065574
hotmail.fr          0.000000
icloud.com          0.031434
juno.com            0.018634
live.com            0.027622
live.com.mx         0.054740
live.fr             0.000000
mac.com             0.032110
mail.com            0.189624
me.com              0.017740
msn.com             0.021994
netzero.com         0.000000
netzero.net         0.005102


In [21]:
train['M4'].value_counts()

M4
M0    196405
M2     59865
M1     52826
Name: count, dtype: int64

## Chapter 5 — M Column Features

### Why
From EDA: M columns are match flags (name, address, email match).
M4 is ordinal (M0/M1/M2) with M2=11.37% fraud rate — 3x baseline.
Missing M values (up to 59% null) are themselves a signal — no match check performed.

### Features Built
- `M4_encoded` — ordinal encoding: M0→0, M1→1, M2→2. Preserves degree of mismatch.
  NaN left as-is — tree models handle natively
- `M_null_count` — number of missing M columns per transaction (0–9).
  High null count = no identity verification performed = fraud signal

In [14]:
train['M4_encoded'] = train['M4'].map({'M0': 0, 'M1': 1, 'M2': 2})

m_cols = [col for col in train.columns if col.startswith('M')]
train['M_null_count'] = train[m_cols].isnull().sum(axis=1)


In [15]:
train['M_null_count']

0          3
1          6
2          0
3          6
4         10
          ..
590535     0
590536     0
590537     6
590538     3
590539     3
Name: M_null_count, Length: 590540, dtype: int64

## Chapter 6 — Velocity Feature (C Columns)

### Why
From EDA: C columns are masked counting features (addresses, transactions per card etc.).
Fraudsters rack up high counts before a stolen card is blocked.
Individual C columns are weak — but combined velocity is a stronger signal.

### Features Built
- `high_velocity` — count of C columns above their 75th percentile for each transaction
  Range: 0–14. High value = card being used abnormally across multiple counting dimensions

In [16]:
c_cols = [col for col in train.columns if col.startswith('C')]
c_75th = train[c_cols].quantile(0.75)
train['high_velocity'] = (train[c_cols] > c_75th).sum(axis=1)


In [17]:
train['high_velocity']

0         0
1         0
2         0
3         3
4         2
         ..
590535    0
590536    0
590537    0
590538    2
590539    0
Name: high_velocity, Length: 590540, dtype: int64

In [20]:
new_features = ['log_TransactionAmt', 'is_outlier', 'hour', 'day_of_week', 
                'hour_is_risky', 'card4_6', 'email_target_encoded',
                'M4_encoded', 'M_null_count', 'high_velocity']

print(train[new_features].head())
print("\nNull counts:")
print(train[new_features].isnull().sum())
print("\nValue ranges:")
print(train[new_features].describe().T[['min','max','mean']])


   log_TransactionAmt  is_outlier  hour  day_of_week  hour_is_risky  \
0            4.241327           0     0            1              0   
1            3.401197           0     0            1              0   
2            4.094345           0     0            1              0   
3            3.931826           0     0            1              0   
4            3.931826           0     0            1              0   

             card4_6  email_target_encoded  M4_encoded  M_null_count  \
0    discover_credit              0.034990         2.0             3   
1  mastercard_credit              0.043542         0.0             6   
2         visa_debit              0.094584         0.0             0   
3   mastercard_debit              0.022757         0.0             6   
4  mastercard_credit              0.043542         NaN            10   

   high_velocity  
0              0  
1              0  
2              0  
3              3  
4              2  

Null counts:
log_Transact

## Chapter 7 — Missing Value Imputation

### Strategy
- **Numeric columns** → median imputation. Median is robust to outliers (mean is not).
  With TransactionAmt reaching $31,937, mean imputation would bias missing values upward
- **Categorical columns** → fill with string `"missing"`. Preserves missingness as
  an explicit category the model can learn from — better than dropping or guessing
- **High-null columns (D, id_)** → NOT dropped. Tree models handle sparse features well
  and missingness itself may carry signal (e.g. no device fingerprint = higher fraud risk)

In [28]:
numeric_cols=train.select_dtypes(include=np.number).columns.tolist()
cat_cols=train.select_dtypes(include='object').columns.tolist()


train[numeric_cols]=train[numeric_cols].fillna(train[numeric_cols].median())
train[cat_cols]=train[cat_cols].fillna('unknown')


## Chapter 8 — Time-Based Train/Val Split

### Why time-based (not random)
In production, the model always predicts future transactions from past patterns.
Random splitting leaks future fraud patterns into training — validation AUC is
artificially inflated and does not reflect real-world performance.

Fraud patterns also drift over time (1.85%–5.06% weekly range from EDA) —
random split would hide this drift.

### Split
- Train: first 75% of `TransactionDT` — 442,905 rows, 3.51% fraud rate
- Val: last 25% of `TransactionDT` — 147,635 rows, 3.45% fraud rate
- Fraud rates nearly identical ✅ — no temporal skew

In [ ]:
# Time-based split — NOT random
split_point = train['TransactionDT'].quantile(0.75)

train_df = train[train['TransactionDT'] <= split_point].copy()
val_df   = train[train['TransactionDT'] >  split_point].copy()

print(f"Train size: {len(train_df):,} ({len(train_df)/len(train)*100:.1f}%)")
print(f"Val size:   {len(val_df):,} ({len(val_df)/len(train)*100:.1f}%)")
print(f"Train fraud rate: {train_df['isFraud'].mean()*100:.2f}%")
print(f"Val fraud rate:   {val_df['isFraud'].mean()*100:.2f}%")


Train size: 442,905 (75.0%)
Val size:   147,635 (25.0%)
Train fraud rate: 3.51%
Val fraud rate:   3.45%


## Chapter 9 — Encoding

### Target Encoding (recomputed on train only)
Fixes leakage from Chapter 4. `email_target_encoded` and `card4_card6_encoded`
now fitted on train_df only and applied to both train and val.

### Label Encoding
Low cardinality categoricals (ProductCD, card4, card6, M columns) → integer labels.
`LabelEncoder.fit_transform` on train, `.transform` only on val —
fitting on val would leak val category distribution into the encoder.

In [ ]:
# Recompute target encodings on train only — fixes leakage
email_fraud_map = train_df.groupby('P_emaildomain')['isFraud'].mean()
card_fraud_map  = train_df.groupby('card4_6')['isFraud'].mean()

for df in [train_df, val_df]:
    df['email_target_encoded'] = df['P_emaildomain'].map(email_fraud_map).fillna(train_df['isFraud'].mean())
    df['card4_6_target_encoded'] = df['card4_6'].map(card_fraud_map).fillna(train_df['isFraud'].mean())

In [35]:
from sklearn.preprocessing import LabelEncoder

low_card_cols = ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']

le = LabelEncoder()
for col in low_card_cols:
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    val_df[col]   = le.transform(val_df[col].astype(str))


In [36]:
import json

# Save features
train_df.to_parquet('data/processed/train_features.parquet')   # data/processed/train_features.parquet
val_df.to_parquet('data/processed/val_features.parquet')     # data/processed/val_features.parquet

# Save feature list
feature_cols = [col for col in train_df.columns if col not in ['TransactionID', 'isFraud', 'TransactionDT']]
with open('data/processed/feature_names.json', 'w') as f:
    json.dump(feature_cols, f)

print(f"Train saved: {train_df.shape}")
print(f"Val saved:   {val_df.shape}")
print(f"Features:    {len(feature_cols)}")


Train saved: (442905, 446)
Val saved:   (147635, 446)
Features:    443
